# Project: CICE Configuration - Ayy, It's Chilly!

This project is about setting up a CICE configuration and any nuances we might want to consider.

Download this project by either:

1. Running the CrocoDash CLI command: `crocodash template --machine derecho --notebook crocodash.projects.cice`
2. Copying the file from your CrocoDash checkout: `demos/crocodash/projects/cice.ipynb`

Grid Generation - The first thing we are going to do is think about how to get a CICE domain going - either pretty northerly or southerly. 

Given how lat/lon changes a lot up there, we'll use a projected grid. The only nuance is in the forcing, so we'll rip through a few steps first 

In [ ]:
from pathlib import Path

from CrocoDash.grid import Grid
from CrocoDash.vgrid import VGrid
from CrocoDash.topo import Topo
from CrocoDash.case import Case


grid = Grid.from_projection(
    crs="EPSG:3995",
    x_min=-1_000_000,  # meters from pole
    x_max= 1_000_000,
    y_min=-1_000_000,
    y_max= 1_000_000,
    resolution_m=50_000,  # 50 km
    name="arctic_50km",
)

topo = Topo(
    grid = grid,
    min_depth = 9.5, # in meters
)

topo.set_from_dataset(    
    bathymetry_path = "<GEBCO_LOWRES>",
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation"
)

topo.depth.plot()
vgrid  = VGrid.hyperbolic(
    nk = 75, # number of vertical levels
    depth = topo.max_depth,
    ratio=20.0 # target ratio of top to bottom layer thicknesses
)


In [ ]:
# CESM case (experiment) name
casename = "arctic_50km"

# CESM source root (Update this path accordingly!!!)
cesmroot ="<CESM>"

# Place where all your input files go 
inputdir = Path("<inputdir>") / casename
    
# CESM case directory
caseroot = Path("<casedir>") / casename


case = Case(
    cesmroot = cesmroot,
    caseroot = caseroot,
    inputdir = inputdir,
    ocn_grid = grid,
    ocn_vgrid = vgrid,
    ocn_topo = topo,
    project = '<PROJECT>',
    override = True,
    machine = "derecho",
    compset = "GR_JRA" )


## CICE OBC Forcing

CICE forcings "restore" from a grid that is exactly one extra cell in each direction. The cice forcing product generates an halo grid (by one cell) that CICE restores from. At the current moment, we use a restart, or collection of restarts as the data that goes on the halo'd grid. So we need to collect a folder of restarts or at least one restart we can point the CrocoDash cice forcing function at. Currently, since this is a new CICE feature, this is the primary way to do open boundary conditions. In the following cell, we use the only cice forcing method and point it at exactly one restart.

In [ ]:
case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    function_name="get_glorys_data_from_rda",
    cice_product_name="cice_restart",              # default
    cice_function_name="get_cice_restart_subset",  # default
    cice_function_args={
        "restart_path": "PATH/TO/global_cice_restart.cice.r.2020-01-01-00000.nc",
        "grid_path": "PATH/TO/cice_grid.nc",  # the restart's companion grid file
    },
    n_halo_cells=1,                    
)


In [ ]:
case.process_forcings()


This restoring we are setting up here is only for boundary conditions. If you want to start from an initial condition, check out the docs here: https://crocodile-cesm.github.io/CrocoGallery/latest/crocodash/case-setup/#optional-warm-start-from-cice-restart-files

I'd then mess with the NTASKS/ROOTPE layout to make sure everything looks reasonable, then a quick build & submit.

Feel free to iterate with process forcings! You can run it on the command line with `crocodash process --caseroot YOURCASEROOT`!